# 📖 Notebook 4: Data Retention & Purging

Data retention answers one critical question: **"How long should we keep this data?"** Every day you store personal data is another day it could be breached, subpoenaed, or misused. In this notebook, we'll build automated retention policies that delete or anonymize data when its time is up.

## Learning Objectives

By the end of this notebook, you'll understand:
- How to define retention policies per data type
- How to implement hard delete, soft delete, and anonymization purge strategies
- How to build an audit trail that proves compliance
- How to automate purging with a scheduled job
- Why audit logs are legally required (even after data is deleted)

## 🛠️ Setup

```bash
cd 08-enterprise/privacy-review
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
from datetime import datetime, timedelta

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "privacy_review",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 📋 Step 1: Understand Our Retention Policies

Let's look at the retention policies we set up in our database. Each policy defines:
- **Which table** the policy applies to
- **How many days** data can be kept
- **What strategy** to use when the time is up (hard delete, soft delete, or anonymize)
- **Legal basis** — why this retention period was chosen

In [ ]:
def load_retention_policies():
    """Load all active retention policies from the database."""
    conn = get_db_connection()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cursor.execute("""
        SELECT table_name, retention_days, description,
               legal_basis, purge_strategy, is_active
        FROM data_retention_policies
        WHERE is_active = TRUE
        ORDER BY retention_days
    """)

    policies = [dict(row) for row in cursor.fetchall()]
    conn.close()
    return policies

policies = load_retention_policies()

print("📋 Active Data Retention Policies")
print("=" * 90)
print(f"  {'Table':<20} {'Days':>6} {'Strategy':<12} {'Legal Basis':<35} {'Description'}")
print("-" * 90)

strategy_icons = {"hard_delete": "🗑️", "soft_delete": "🏷️", "anonymize": "🔒"}
for p in policies:
    icon = strategy_icons.get(p["purge_strategy"], "?")
    days_label = str(p["retention_days"]) if p["retention_days"] > 0 else "immediate"
    print(f"  {p['table_name']:<20} {days_label:>6} {icon} {p['purge_strategy']:<10} "
          f"{p['legal_basis'][:34]:<35} {p['description']}")

print(f"\n💡 {len(policies)} active policies loaded")
print(f"   Strategies: 🗑️ hard_delete (remove rows), 🏷️ soft_delete (mark deleted), 🔒 anonymize (replace PII)")

## ⏱️ Step 1b: Which Clock Does Each Policy Run On?

Every policy above names an event: activity logs age from when they were
written, support tickets are kept "1 year **after resolution**", deleted
accounts are purged "after a 30-day **grace period**". Those are three different
clocks, and using `created_at` for all of them is not a simplification — it is a
different policy that happens to compile.

Get it wrong in the safe direction and you keep data too long. Get it wrong in
the unsafe direction and you delete data that was still inside its window: an
account created two years ago and marked deleted this morning has `created_at`
far in the past, so a `created_at` clock erases it *immediately* and the 30-day
grace period the policy promised never happens.

So we declare the clock per table, and prepare some demo data old enough for the
policies to actually fire. The seeded database is only a few hundred days old,
which is younger than most of these retention periods — without this step the
purge engine below would run, report zeros, and prove nothing.


In [ ]:
# The column (or expression) whose value starts each retention clock.
RETENTION_CLOCK = {
    "activity_log":    "created_at",                          # age of the log line
    "orders":          "COALESCE(delivered_at, created_at)",  # age of the transaction
    "support_tickets": "resolved_at",             # policy says "1 year after resolution"
    "users":           "deletion_requested_at",   # the GDPR grace period starts here
    "payment_methods": "created_at",              # retention_days = 0, never age-scanned
}

def retention_clock(table):
    """The clock column for a table, defaulting to created_at."""
    return RETENTION_CLOCK.get(table, "created_at")

conn = get_db_connection()
cursor = conn.cursor()

# `deletion_requested_at` ships in db/init.sql alongside this notebook. The
# ALTER keeps things working against a database volume created before it.
cursor.execute("ALTER TABLE users ADD COLUMN IF NOT EXISTS deletion_requested_at TIMESTAMP")

# ── Demo fixtures ─────────────────────────────────────────────────────────
# The seeded database is only a few hundred days old — younger than most of
# these retention periods. Without rows old enough to expire, the purge engine
# below would run, report zeros, and prove nothing.
#
# We clear the fixtures from any previous run first so the notebook always
# starts from the same place. Everything deleted here is already past its
# retention window; the engine would remove or redact it anyway.
cursor.execute("DELETE FROM activity_log WHERE created_at < NOW() - INTERVAL '90 days'")
cursor.execute("DELETE FROM support_tickets WHERE created_at < NOW() - INTERVAL '395 days'")
cursor.execute("DELETE FROM orders WHERE created_at < NOW() - INTERVAL '2555 days'")

# Deletion requests at staggered ages: some past the 30-day grace period, some
# still inside it. Refreshed every run so the demo is reproducible.
cursor.execute("""
    UPDATE users
    SET deletion_requested_at = NOW() - ((id / 6) * 12 * INTERVAL '1 day')
    WHERE account_status = 'deleted'
""")
erasure_requests = cursor.rowcount

cursor.execute("""
    INSERT INTO activity_log (user_id, action, resource_type, resource_id, ip_address,
                              user_agent, geo_country, geo_city, created_at)
    SELECT u.id, 'page_view', 'product', i,
           '10.0.0.' || (i % 250)::text, 'RETENTION-DEMO', 'US', 'Seattle',
           NOW() - ((91 + i) * INTERVAL '1 day')
    FROM generate_series(1, 120) AS i,
         (SELECT id FROM users WHERE account_status = 'active' ORDER BY id LIMIT 1) u
""")
demo_activity = cursor.rowcount

cursor.execute("""
    INSERT INTO orders (user_id, order_number, shipping_name, shipping_address,
                        shipping_city, shipping_state, shipping_zip, shipping_country,
                        subtotal, tax, total, status, created_at, delivered_at)
    SELECT u.id, 'RD-' || to_char(NOW(), 'YYMMDDHH24MISS') || i::text,
           u.first_name || ' ' || u.last_name, u.street_address,
           u.city, u.state, u.zip_code, u.country,
           100.00, 8.00, 108.00, 'delivered',
           NOW() - ((2600 + i * 30) * INTERVAL '1 day'),
           NOW() - ((2590 + i * 30) * INTERVAL '1 day')
    FROM generate_series(1, 12) AS i,
         (SELECT id, first_name, last_name, street_address, city, state, zip_code, country
          FROM users WHERE account_status = 'active' ORDER BY id LIMIT 1) u
""")
demo_orders = cursor.rowcount

cursor.execute("""
    INSERT INTO support_tickets (user_id, subject, description, internal_notes,
                                 priority, status, created_at, resolved_at)
    SELECT u.id,
           'RETENTION-DEMO resolved ticket ' || i,
           'My card ending 1234 was charged twice. Reach me at demo' || i || '@example.com.',
           'Verified via SSN last 4: 9876. Refund issued.',
           'medium', 'resolved',
           NOW() - ((420 + i * 10) * INTERVAL '1 day'),
           NOW() - ((400 + i * 10) * INTERVAL '1 day')
    FROM generate_series(1, 4) AS i,
         (SELECT id FROM users WHERE account_status = 'active' ORDER BY id LIMIT 1) u
""")
demo_tickets = cursor.rowcount

# One very old ticket that is still open — its retention clock never started.
cursor.execute("""
    INSERT INTO support_tickets (user_id, subject, description, internal_notes,
                                 priority, status, created_at, resolved_at)
    SELECT u.id,
           'RETENTION-DEMO never resolved',
           'Still waiting. My email is forgotten@example.com, DOB 01/02/1970.',
           NULL, 'low', 'open',
           NOW() - (900 * INTERVAL '1 day'), NULL
    FROM (SELECT id FROM users WHERE account_status = 'active' ORDER BY id LIMIT 1) u
""")
demo_open_ticket = cursor.rowcount

conn.commit()
conn.close()

print("⏱️ Retention clocks")
print("=" * 78)
for table, clock in RETENTION_CLOCK.items():
    print(f"  {table:<18} → {clock}")

print("\n🧪 Demo fixtures prepared (this cell resets them, so re-running is safe)")
print(f"   {erasure_requests} deletion requests dated across the 30-day grace period")
print(f"   +{demo_activity} activity rows aged past 90 days")
print(f"   +{demo_orders} orders aged past 7 years")
print(f"   +{demo_tickets} resolved tickets aged past 1 year")
print(f"   +{demo_open_ticket} very old ticket that is still OPEN (clock never started)")

assert erasure_requests > 0, (
    "no accounts are marked deleted — notebook 4 has already erased them all. "
    "Recreate the database with `docker compose down -v && docker compose up -d`."
)


## 🔍 Step 2: Find Data That Has Expired

Before we can purge anything, we need to identify which records have exceeded their retention period. We'll write a scanner that checks each table against its policy.

**Important**: each table is scanned against the clock we declared above, not
against `created_at` for everything. The scan and the purge must use the *same*
clock and the *same* filters — a scan that counts rows the purge would not touch
is a report that quietly lies to whoever reads it.

Watch the last column: rows whose clock is NULL (an unresolved support ticket)
never start their retention period at all. That is not a bug in the code, it is
a real and common way personal data outlives its policy — and the scan should
say so out loud rather than leave it out of the count.

In [ ]:
def find_expired_records(policy):
    """Find records that have exceeded their retention period."""
    table = policy["table_name"]
    retention_days = policy["retention_days"]
    clock = retention_clock(table)

    if retention_days == 0:
        # Immediate deletion policies are handled differently
        return {"table": table, "expired_count": 0, "total_count": 0,
                "oldest_record": None, "note": "Immediate deletion — handled on user action"}

    cutoff_date = datetime.now() - timedelta(days=retention_days)

    # `users` is only purged when the account is actually marked deleted. The
    # clock (deletion_requested_at) is already specific to erasure requests, but
    # a user who asked to be deleted and then changed their mind can have a
    # stale timestamp — the status check is the belt to that pair of braces.
    # The scan must apply exactly the filter the purge will apply, or the counts
    # reported here describe rows that never get touched.
    extra_where = "AND account_status = 'deleted'" if table == "users" else ""

    conn = get_db_connection()
    cursor = conn.cursor()

    # Count expired records
    cursor.execute(
        f'SELECT COUNT(*) FROM "{table}" WHERE {clock} < %s {extra_where}',
        (cutoff_date,)
    )
    expired_count = cursor.fetchone()[0]

    # Count total records
    cursor.execute(f'SELECT COUNT(*) FROM "{table}"')
    total_count = cursor.fetchone()[0]

    # Rows whose retention clock never started. These are invisible to the
    # purge and will sit there forever unless somebody goes looking.
    cursor.execute(f'SELECT COUNT(*) FROM "{table}" WHERE {clock} IS NULL')
    unclocked = cursor.fetchone()[0]

    # Oldest record *by the clock this policy runs on*
    cursor.execute(f'SELECT MIN({clock}) FROM "{table}"')
    oldest = cursor.fetchone()[0]

    conn.close()

    return {
        "table": table,
        "clock": clock,
        "expired_count": expired_count,
        "total_count": total_count,
        "unclocked": unclocked,
        "cutoff_date": cutoff_date,
        "oldest_record": oldest,
        "retention_days": retention_days,
        "strategy": policy["purge_strategy"]
    }

# Scan all tables
print("🔍 Retention Scan Results")
print("=" * 104)
print(f"  {'Table':<18} {'Clock':<34} {'Total':>6} {'Expired':>8} {'No clock':>9} {'Strategy':<11} {'Cutoff'}")
print("-" * 104)

scan_results = []
for policy in policies:
    result = find_expired_records(policy)
    scan_results.append(result)

    if "note" in result:
        print(f"  {result['table']:<18} {'—':<34} {'—':>6} {'—':>8} {'—':>9} "
              f"{'—':<11} {result['note']}")
        continue

    pct = (result["expired_count"] / result["total_count"] * 100) if result["total_count"] > 0 else 0
    icon = "🔴" if pct > 50 else "🟡" if pct > 20 else "🟢"
    flag = "⚠️" if result["unclocked"] else " "
    print(f"  {result['table']:<18} {result['clock']:<34} {result['total_count']:>6} "
          f"{icon}{result['expired_count']:>6} {flag}{result['unclocked']:>7} "
          f"{result['strategy']:<11} {result['cutoff_date'].strftime('%Y-%m-%d')}")

total_expired = sum(r["expired_count"] for r in scan_results if "expired_count" in r)
total_unclocked = sum(r.get("unclocked", 0) for r in scan_results)
print(f"\n⚠️  Total records past retention: {total_expired}")
print(f"   These should be purged to comply with retention policies.")
unclocked_by_table = {r["table"]: r.get("unclocked", 0) for r in scan_results}
print(f"⏸️  Records whose retention clock has not started: {total_unclocked}")
print(f"   {unclocked_by_table.get('users', 0)} users — normal: no erasure request means no clock.")
print(f"   {unclocked_by_table.get('support_tickets', 0)} support tickets — not normal: an unresolved ticket")
print(f"   is kept forever by a policy that reads 'one year after resolution'.")
print(f"   Nothing in the purge will ever see it. Someone has to own that queue.")

# The rest of the notebook demonstrates three purge strategies. If nothing is
# expired, it demonstrates nothing — and every 'after' cell below would happily
# print zeros under a heading that claims otherwise.
expired_by_table = {r["table"]: r["expired_count"] for r in scan_results if "expired_count" in r}
assert expired_by_table.get("activity_log", 0) > 0, "no expired activity logs to hard-delete"
assert expired_by_table.get("support_tickets", 0) > 0, "no expired tickets to anonymize"
assert expired_by_table.get("orders", 0) > 0, "no expired orders to anonymize"
assert expired_by_table.get("users", 0) > 0, "no erasure requests past the grace period"
assert unclocked_by_table.get("support_tickets", 0) > 0, (
    "the never-resolved ticket fixture is missing — the 'clock never starts' "
    "failure mode has nothing to show"
)
print("\n✅ Every policy has work to do — the purge below can actually be observed")


## 🗑️ Step 3: Implement Purge Strategies

We have three purge strategies. Each handles expired data differently:

1. **Hard Delete** — remove the rows entirely (for data with no legal hold)
2. **Soft Delete** — mark as deleted but keep in DB (for grace periods)
3. **Anonymize** — replace PII with placeholder values but keep the record (for analytics)

All strategies must write to the **audit log** so we can prove to regulators what was deleted and why.

### Erasure is not one DELETE

Notebook 1 found the same person's name and address duplicated into
`orders.shipping_*`, their IP address in `activity_log`, and free-text copies in
support tickets. Deleting the row in `users` leaves all of that behind — the
foreign keys are `ON DELETE SET NULL`, so the copies survive with the link
removed, which reads as "erased" on a schema diagram and is nothing of the sort.

So the erasure path below scrubs the copies **first**, in the same transaction,
and then verifies that no PII belonging to those subjects is still reachable
before it reports success.

In [ ]:
def log_purge_action(cursor, table_name, records_affected, strategy, reason):
    """Write to the purge audit log. This is legally required."""
    cursor.execute("""
        INSERT INTO purge_audit_log
            (table_name, records_affected, purge_strategy, purge_reason, executed_by)
        VALUES (%s, %s, %s, %s, %s)
    """, (table_name, records_affected, strategy, reason, "retention-engine"))


def purge_hard_delete(table_name, retention_days, extra_where=None):
    """Hard delete: remove rows past their retention period.

    Ages rows by the table's declared retention clock, not by `created_at`.
    `extra_where` is an optional SQL fragment (no parameters) that further
    restricts which rows are eligible.

    The audited count comes from `cursor.rowcount` — what the DELETE actually
    removed — rather than a COUNT(*) taken beforehand. An audit log is evidence;
    it should record what happened, not what we predicted would happen.
    """
    clock = retention_clock(table_name)
    cutoff = datetime.now() - timedelta(days=retention_days)

    where_sql = f'{clock} < %s'
    if extra_where:
        where_sql += f' AND ({extra_where})'

    conn = get_db_connection()
    cursor = conn.cursor()

    # Delete the records. FK columns on dependent tables are declared with
    # ON DELETE CASCADE (payment_methods) or ON DELETE SET NULL (orders,
    # support_tickets, activity_log) so this DELETE cannot be blocked by a
    # foreign-key violation.
    cursor.execute(f'DELETE FROM "{table_name}" WHERE {where_sql}', (cutoff,))
    count = cursor.rowcount

    if count > 0:
        log_purge_action(
            cursor, table_name, count, "hard_delete",
            f"Retention policy: {retention_days} days on {clock}. "
            f"Cutoff: {cutoff.isoformat()}. Filter: {extra_where or 'none'}"
        )

    conn.commit()
    conn.close()
    return count


def purge_anonymize(table_name, retention_days, pii_columns):
    """Anonymize: replace PII columns with placeholders but keep the row.

    Skips rows that are already anonymized. Without that guard every run
    re-writes the same rows and logs them again, and the audit trail ends up
    claiming we anonymized 40 records every night forever — which makes the one
    number a regulator actually cares about worthless.
    """
    clock = retention_clock(table_name)
    cutoff = datetime.now() - timedelta(days=retention_days)

    conn = get_db_connection()
    cursor = conn.cursor()

    set_sql = ", ".join(f'"{col}" = %s' for col in pii_columns)
    # IS DISTINCT FROM, not <>: a NULL column is still "not yet anonymized".
    not_yet = " OR ".join(f'"{col}" IS DISTINCT FROM %s' for col in pii_columns)
    values = list(pii_columns.values())

    cursor.execute(
        f'UPDATE "{table_name}" SET {set_sql} WHERE {clock} < %s AND ({not_yet})',
        values + [cutoff] + values
    )
    count = cursor.rowcount

    if count > 0:
        log_purge_action(
            cursor, table_name, count, "anonymize",
            f"Anonymized columns: {list(pii_columns.keys())} on rows with "
            f"{clock} < {cutoff.isoformat()}"
        )

    conn.commit()
    conn.close()
    return count


# Copies of a subject's PII that live outside the `users` row. Every one of
# these tables keeps its rows for its own reasons (tax law, support history,
# security forensics) — they keep the row, they do not get to keep the person.
SUBJECT_PII_COPIES = {
    "orders": {
        "shipping_name": "[ERASED — data subject request]",
        "shipping_address": "[ERASED]",
        "shipping_city": "[ERASED]",
        "shipping_state": "[ERASED]",
        "shipping_zip": "[ERASED]",
    },
    "support_tickets": {
        "description": "[ERASED — data subject request]",
        "internal_notes": "[ERASED — data subject request]",
    },
    "activity_log": {
        "ip_address": "[ERASED]",
        "user_agent": "[ERASED]",
        "geo_city": "[ERASED]",
    },
}


def erase_subject_pii(cursor, user_ids):
    """Scrub every copy of these users' PII from the tables we are keeping."""
    scrubbed = {}
    for table, columns in SUBJECT_PII_COPIES.items():
        set_sql = ", ".join(f'"{col}" = %s' for col in columns)
        cursor.execute(
            f'UPDATE "{table}" SET {set_sql} WHERE user_id = ANY(%s)',
            list(columns.values()) + [user_ids]
        )
        scrubbed[table] = cursor.rowcount
    return scrubbed


def count_residual_pii(cursor, user_ids):
    """Rows still holding these subjects' PII. Must be zero after an erasure."""
    residual = 0
    for table, columns in SUBJECT_PII_COPIES.items():
        first_col = next(iter(columns))
        # Any '[...]' placeholder counts as scrubbed: retention may already have
        # written '[REDACTED — retention expired]' into the same column before
        # the erasure request arrived.
        cursor.execute(
            f'SELECT COUNT(*) FROM "{table}" '
            f'WHERE user_id = ANY(%s) AND "{first_col}" IS NOT NULL '
            f"AND \"{first_col}\" NOT LIKE '[%%'",
            (user_ids,)
        )
        residual += cursor.fetchone()[0]
    return residual


def purge_user_erasure(retention_days):
    """GDPR Art. 17 erasure for accounts past the grace period.

    Order matters: scrub the copies, verify nothing is left, and only then
    delete the `users` row. Delete first and the `ON DELETE SET NULL` foreign
    keys cut the link, leaving orphaned rows full of somebody's name and address
    that we can no longer even find.
    """
    clock = retention_clock("users")
    cutoff = datetime.now() - timedelta(days=retention_days)

    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute(
        f"SELECT id FROM users WHERE {clock} < %s AND account_status = 'deleted'",
        (cutoff,)
    )
    subject_ids = [row[0] for row in cursor.fetchall()]

    if not subject_ids:
        conn.close()
        return {"deleted": 0, "scrubbed": {}, "residual": 0, "subject_ids": []}

    scrubbed = erase_subject_pii(cursor, subject_ids)
    residual = count_residual_pii(cursor, subject_ids)

    cursor.execute("DELETE FROM users WHERE id = ANY(%s)", (subject_ids,))
    deleted = cursor.rowcount

    # The audit entry carries the subject ids on purpose. "We deleted 6 rows"
    # cannot answer "prove you erased *me*"; a pseudonymous id can, and it is
    # not a name, an email or an address.
    log_purge_action(
        cursor, "users", deleted, "hard_delete",
        f"GDPR Art. 17 erasure. Clock: {clock} < {cutoff.isoformat()} "
        f"({retention_days}-day grace period). Subject ids: {subject_ids}. "
        f"PII copies scrubbed: {scrubbed}. Residual PII rows: {residual}"
    )

    conn.commit()
    conn.close()
    return {"deleted": deleted, "scrubbed": scrubbed,
            "residual": residual, "subject_ids": subject_ids}


print("✅ Purge strategies loaded")
print("   - purge_hard_delete():  removes rows entirely, audited by rowcount")
print("   - purge_anonymize():    replaces PII in place, skips already-done rows")
print("   - purge_user_erasure(): scrubs every copy of a subject's PII, then deletes")


## ⚡ Step 4: Run the Purge Engine

Now let's execute the purges. We'll show before/after counts and what the audit log captures.

In [ ]:
# Before purge — snapshot current state
conn = get_db_connection()
cursor = conn.cursor()

tables = ["activity_log", "support_tickets", "orders", "users"]
before_counts = {}
for table in tables:
    cursor.execute(f'SELECT COUNT(*) FROM "{table}"')
    before_counts[table] = cursor.fetchone()[0]

# Rows still carrying PII, so we can show the anonymize strategies working.
cursor.execute("SELECT COUNT(*) FROM orders WHERE shipping_name NOT LIKE '[%'")
before_counts["orders_with_pii"] = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM support_tickets WHERE description NOT LIKE '[%'")
before_counts["tickets_with_pii"] = cursor.fetchone()[0]

conn.close()

print("📊 Before Purge — Record Counts")
print("=" * 46)
for table in tables:
    print(f"  {table:<20} {before_counts[table]:>6} records")
print(f"  {'orders w/ PII':<20} {before_counts['orders_with_pii']:>6}")
print(f"  {'tickets w/ PII':<20} {before_counts['tickets_with_pii']:>6}")

print("\n🔄 Running purge engine...\n")


In [ ]:
# Execute purges according to each policy

# Which columns each anonymize policy replaces
anonymize_columns = {
    "support_tickets": {
        "description": "[REDACTED — retention expired]",
        "internal_notes": "[REDACTED — retention expired]",
        "subject": "[Support ticket — anonymized]"
    },
    "orders": {
        "shipping_name": "[REDACTED]",
        "shipping_address": "[REDACTED]",
        "shipping_city": "[REDACTED]",
        "shipping_state": "[REDACTED]",
        "shipping_zip": "[REDACTED]"
    }
}

purge_results = []
erasure = None

for policy in policies:
    table = policy["table_name"]
    strategy = policy["purge_strategy"]
    days = policy["retention_days"]

    if days == 0:
        continue  # Skip immediate-deletion policies

    if table == "users":
        # Erasure is not a DELETE. Scrub every copy of the subject's PII from
        # the tables we keep, then remove the account row.
        erasure = purge_user_erasure(days)
        purge_results.append({"table": table, "strategy": strategy,
                              "purged": erasure["deleted"]})
    elif strategy == "hard_delete":
        count = purge_hard_delete(table, days)
        purge_results.append({"table": table, "strategy": strategy, "purged": count})
    elif strategy == "anonymize" and table in anonymize_columns:
        count = purge_anonymize(table, days, anonymize_columns[table])
        purge_results.append({"table": table, "strategy": strategy, "purged": count})

# Display results
print("🗑️ Purge Results")
print("=" * 60)
print(f"  {'Table':<20} {'Strategy':<14} {'Records Affected':>18}")
print("-" * 60)

for r in purge_results:
    icon = "🗑️" if r["strategy"] == "hard_delete" else "🔒"
    print(f"  {r['table']:<20} {icon} {r['strategy']:<12} {r['purged']:>18}")

total_purged = sum(r["purged"] for r in purge_results)
print(f"\n  Total records processed: {total_purged}")

# ── Erasure detail: what a DELETE on `users` does NOT do by itself ─────────
print("\n🧹 Data subject erasure detail")
print("-" * 60)
print(f"  Accounts erased:       {erasure['deleted']} (ids {erasure['subject_ids']})")
for table, rows in erasure["scrubbed"].items():
    print(f"  PII copies scrubbed:   {rows:>4} rows in {table}")
print(f"  Residual PII rows:     {erasure['residual']}")

assert erasure is not None, "the users retention policy is missing or inactive"
assert erasure["deleted"] > 0, (
    "no accounts were past the 30-day grace period. If you have run this "
    "notebook before, those accounts are already erased — recreate the database "
    "with `docker compose down -v && docker compose up -d` to see it again."
)
assert erasure["residual"] == 0, (
    f"{erasure['residual']} rows still hold the erased subjects' PII — deleting "
    "the users row is not erasure while the copies survive"
)
assert sum(erasure["scrubbed"].values()) > 0, (
    "the erased users had orders, tickets and activity rows; if nothing was "
    "scrubbed then the copies were never being cleaned up"
)

# The grace period has to protect somebody, or it is not a grace period.
users_policy = next(p for p in policies if p["table_name"] == "users")
grace_cutoff = datetime.now() - timedelta(days=users_policy["retention_days"])
conn = get_db_connection()
cursor = conn.cursor()
cursor.execute("""
    SELECT COUNT(*) FROM users
    WHERE account_status = 'deleted' AND deletion_requested_at >= %s
""", (grace_cutoff,))
still_in_grace = cursor.fetchone()[0]
conn.close()

assert still_in_grace > 0, (
    "every deleted account was erased — with a created_at clock that is exactly "
    "what happens, and the 30-day recovery window the policy promises does not "
    "exist"
)
print(f"  Still inside the grace window (untouched): {still_in_grace} accounts ✅")

# Running the same policy twice must be a no-op. If it is not, the audit log
# inflates every night and stops being evidence of anything.
tickets_policy = next(p for p in policies if p["table_name"] == "support_tickets")
repeat = purge_anonymize("support_tickets", tickets_policy["retention_days"],
                         anonymize_columns["support_tickets"])
assert repeat == 0, (
    f"re-running the anonymize policy touched {repeat} rows again — the "
    "already-anonymized guard is not working"
)
print(f"  Re-running the anonymize policy touched {repeat} rows ✅ (idempotent)")


In [ ]:
# After purge — compare counts
conn = get_db_connection()
cursor = conn.cursor()

print("📊 After Purge — Record Counts")
print("=" * 66)
print(f"  {'Table':<20} {'Before':>8} {'After':>8} {'Change':>9}  What happened")
print("-" * 66)

after_counts = {}
for table in tables:
    cursor.execute(f'SELECT COUNT(*) FROM "{table}"')
    after_counts[table] = cursor.fetchone()[0]
    change = after_counts[table] - before_counts[table]
    strategy = next((p["purge_strategy"] for p in policies if p["table_name"] == table), "—")
    if change < 0:
        note = "🗑️ rows deleted"
    elif strategy == "anonymize":
        note = "🔒 rows kept, PII replaced"
    else:
        note = "— nothing expired"
    print(f"  {table:<20} {before_counts[table]:>8} {after_counts[table]:>8} "
          f"{change:>+9}  {note}")

cursor.execute("SELECT COUNT(*) FROM orders WHERE shipping_name NOT LIKE '[%'")
orders_with_pii = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM support_tickets WHERE description NOT LIKE '[%'")
tickets_with_pii = cursor.fetchone()[0]
conn.close()

print(f"\n  {'orders w/ PII':<20} {before_counts['orders_with_pii']:>8} "
      f"{orders_with_pii:>8} {orders_with_pii - before_counts['orders_with_pii']:>+9}")
print(f"  {'tickets w/ PII':<20} {before_counts['tickets_with_pii']:>8} "
      f"{tickets_with_pii:>8} {tickets_with_pii - before_counts['tickets_with_pii']:>+9}")

# Row counts alone cannot tell you whether an anonymize policy did anything —
# the count is identical either way. Check the PII, not the rows.
assert after_counts["activity_log"] < before_counts["activity_log"], (
    "hard_delete on activity_log removed nothing"
)
assert after_counts["orders"] == before_counts["orders"], (
    "anonymize must keep the row — the tax team still needs the transaction"
)
assert orders_with_pii < before_counts["orders_with_pii"], (
    "orders row count is unchanged AND the PII is unchanged: the anonymize "
    "policy did nothing at all"
)
assert tickets_with_pii < before_counts["tickets_with_pii"], (
    "no support ticket text was redacted"
)
print("\n✅ Purge assertions passed")
print("\n💡 'orders' keeps its row count because we anonymized rather than deleted:")
print("   the row survives for the 7-year tax obligation, the name and address do")
print("   not. That distinction — the row versus the person — is the whole of this")
print("   notebook. A masked column in a view would have kept both.")


## 🔍 Step 5: Verify Anonymization

Let's look at the anonymized records to confirm PII was actually removed.

In [ ]:
conn = get_db_connection()
cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Check anonymized orders
cursor.execute("""
    SELECT id, order_number, shipping_name, shipping_address,
           shipping_city, total, status, created_at
    FROM orders
    WHERE shipping_name = '[REDACTED]'
    LIMIT 5
""")

anonymized_orders = cursor.fetchall()

print("🔒 Anonymized Order Records")
print("=" * 90)

if anonymized_orders:
    for o in anonymized_orders:
        print(f"\n  Order {o['order_number']}:")
        print(f"    Shipping Name:    {o['shipping_name']}")
        print(f"    Shipping Address: {o['shipping_address']}")
        print(f"    Shipping City:    {o['shipping_city']}")
        print(f"    Total:            ${float(o['total']):.2f}")  # Financial data kept for tax
        print(f"    Status:           {o['status']}")
        print(f"    Created:          {o['created_at']}")

    print(f"\n💡 PII is gone, but financial totals and order metadata remain.")
    print(f"   This lets the tax team still use the data for compliance.")
else:
    print("  No anonymized orders found — the 7-year policy had nothing to work on.")
    print("  💡 Step 1b seeds orders old enough to trigger it; if this branch ran,")
    print("     that fixture did not land.")

assert anonymized_orders, "the anonymize strategy produced no anonymized orders"
assert all(o["total"] is not None for o in anonymized_orders), (
    "anonymize must keep the non-personal columns — an order with no total is "
    "useless to the tax team the 7-year retention exists for"
)

# Check anonymized support tickets
cursor.execute("""
    SELECT id, subject, description, internal_notes, status, created_at
    FROM support_tickets
    WHERE description = '[REDACTED — retention expired]'
    LIMIT 5
""")

anonymized_tickets = cursor.fetchall()

print(f"\n🔒 Anonymized Support Tickets")
print("=" * 90)

if anonymized_tickets:
    for t in anonymized_tickets:
        print(f"\n  Ticket #{t['id']}:")
        print(f"    Subject:  {t['subject']}")
        print(f"    Content:  {t['description']}")
        print(f"    Notes:    {t['internal_notes']}")
        print(f"    Status:   {t['status']}")
else:
    print("  No anonymized tickets found — nothing was past the 1-year window.")

assert anonymized_tickets, "the anonymize strategy redacted no support tickets"

# Two different placeholders, two different reasons. Retention expiry redacts on
# a schedule; an erasure request redacts on demand, for one person, and has to
# reach rows that are nowhere near their retention limit.
cursor.execute("""
    SELECT COUNT(*) AS n FROM orders WHERE shipping_name LIKE '[ERASED%'
""")
erased_rows = cursor.fetchone()["n"]
print(f"\n🧹 Orders scrubbed by a data subject request (not by age): {erased_rows}")
print("   These orders are only a few months old — the 7-year retention policy")
print("   would not have touched them for years. The erasure request did.")
assert erased_rows > 0, (
    "no orders were scrubbed by the erasure path — the copies of an erased "
    "user's name and address are still sitting in the orders table"
)

# And the thing that is NOT true of masking: the value is gone from the table.
cursor.execute("""
    SELECT COUNT(*) AS n FROM orders
    WHERE shipping_name NOT LIKE '[%' AND user_id IS NULL
""")
orphan_pii = cursor.fetchone()["n"]
assert orphan_pii == 0, (
    f"{orphan_pii} orders have no owner but still carry a shipping name — that "
    "is exactly the residue an erasure that only deletes the users row leaves "
    "behind"
)
print(f"✅ No orphaned rows still carrying a name: {orphan_pii}")

conn.close()

## 📝 Step 6: The Audit Trail

The audit trail is **legally critical**. When a regulator asks "show me proof you deleted this user's data", you need to point to the audit log.

The audit log itself must **never** contain PII — it records *what* was deleted, *when*, *why*, and *how many records*, but NOT the actual data.

### But a count is not proof

"We deleted 6 rows from `users` on 14 March" cannot answer *"prove you erased
**me**"*. Something has to tie the entry to the subject, and the only way to do
that without storing a name is to store the pseudonymous key — the user id. So
the erasure entry below carries the subject ids, what was scrubbed in each other
table, and the residual-PII check that ran before the row was deleted.

Note the tension, because it is real and it does not resolve neatly: a user id
is personal data under GDPR, and here we are deliberately keeping some of it
after an erasure request. The legal basis is Art. 17(3)(b)/(e) — compliance with
a legal obligation and the establishment of legal claims. You cannot prove you
erased someone while retaining nothing at all about them; you keep the minimum
that makes the proof possible, you write down why, and the audit log gets the
same retention policy and access controls as everything else.

In [ ]:
def display_audit_trail():
    """Display the purge audit log."""
    conn = get_db_connection()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cursor.execute("""
        SELECT id, table_name, records_affected, purge_strategy,
               purge_reason, executed_by, executed_at
        FROM purge_audit_log
        ORDER BY executed_at DESC
    """)

    logs = cursor.fetchall()
    conn.close()

    print("📝 Purge Audit Trail")
    print("=" * 100)

    if not logs:
        print("  No purge operations recorded yet.")
        return

    for log in logs:
        strategy_icon = "🗑️" if log["purge_strategy"] == "hard_delete" else "🔒"
        print(f"\n  {strategy_icon} Purge #{log['id']} — {log['executed_at']}")
        print(f"     Table:    {log['table_name']}")
        print(f"     Records:  {log['records_affected']}")
        print(f"     Strategy: {log['purge_strategy']}")
        print(f"     Reason:   {log['purge_reason']}")
        print(f"     By:       {log['executed_by']}")

    print(f"\n  📊 Total purge operations: {len(logs)}")
    print(f"  📊 Total records affected: {sum(l['records_affected'] for l in logs)}")

display_audit_trail()

# The audit trail is the deliverable. If a purge ran and the log does not say
# so, the compliance story is "trust us".
conn = get_db_connection()
cursor = conn.cursor()
cursor.execute("SELECT purge_strategy, COUNT(*) FROM purge_audit_log GROUP BY purge_strategy")
by_strategy = dict(cursor.fetchall())
cursor.execute("""
    SELECT purge_reason FROM purge_audit_log
    WHERE table_name = 'users' AND purge_reason LIKE '%Subject ids%'
    ORDER BY id DESC LIMIT 1
""")
erasure_entry = cursor.fetchone()
conn.close()

assert by_strategy.get("hard_delete", 0) > 0, "hard deletes were not audited"
assert by_strategy.get("anonymize", 0) > 0, "anonymizations were not audited"
assert erasure_entry is not None, (
    "the erasure entry must name its subjects — a bare count cannot answer "
    "'prove you erased me'"
)
print(f"\n✅ Audit assertions passed")
print(f"   Per-subject erasure evidence on file: {erasure_entry[0][:110]}...")

## 🤖 Step 7: Build an Automated Retention Scheduler

In production, purging doesn't happen manually — it runs on a schedule (usually daily). Let's build a complete retention engine that could run as a cron job.

We'll also use Redis to track when the last purge ran, prevent duplicate runs, and cache policy lookups.

In [ ]:
class RetentionEngine:
    """Automated data retention and purging engine.
    
    This would run as a daily cron job in production:
    0 2 * * * python retention_engine.py  # Run at 2 AM daily
    """

    def __init__(self):
        self.redis = get_redis_client()
        # Microseconds, not seconds: the run id is also the lock token, and two
        # runs that start in the same second must not be able to claim each
        # other's lock.
        self.run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

        # PII columns to anonymize per table
        self.anonymize_map = {
            "support_tickets": {
                "description": "[REDACTED — retention expired]",
                "internal_notes": "[REDACTED — retention expired]",
                "subject": "[Support ticket — anonymized]"
            },
            "orders": {
                "shipping_name": "[REDACTED]",
                "shipping_address": "[REDACTED]",
                "shipping_city": "[REDACTED]",
                "shipping_state": "[REDACTED]",
                "shipping_zip": "[REDACTED]"
            }
        }

        # `users` is not a plain hard_delete: erasure has to scrub every copy of
        # the subject's PII before the account row goes, so it gets its own
        # routine rather than an extra WHERE clause.

    def acquire_lock(self):
        """Prevent multiple instances from running simultaneously."""
        # SET with NX (only if not exists) and EX (expire after 1 hour)
        acquired = self.redis.set(
            "retention:lock", self.run_id,
            nx=True, ex=3600
        )
        return acquired

    def release_lock(self):
        """Release the lock — but only if we still hold it.

        A bare DELETE releases whatever lock happens to be there, including one
        that a *different* run acquired after ours expired mid-purge. Then two
        engines delete concurrently and the audit log stops matching reality.
        Compare-and-delete, atomically, or do not bother locking at all.
        """
        script = ("if redis.call('get', KEYS[1]) == ARGV[1] "
                  "then return redis.call('del', KEYS[1]) else return 0 end")
        return self.redis.eval(script, 1, "retention:lock", self.run_id) == 1

    def run(self):
        """Execute the full retention cycle."""
        print(f"🤖 Retention Engine — Run {self.run_id}")
        print("=" * 60)

        # Step 1: Acquire lock
        if not self.acquire_lock():
            print("❌ Another retention run is in progress. Exiting.")
            return []
        print("🔒 Lock acquired")

        try:
            # Step 2: Load policies
            policies = load_retention_policies()
            print(f"📋 Loaded {len(policies)} policies")

            results = []

            # Step 3: Execute each policy
            for policy in policies:
                table = policy["table_name"]
                strategy = policy["purge_strategy"]
                days = policy["retention_days"]

                if days == 0:
                    continue

                print(f"\n  Processing: {table} (strategy={strategy}, retention={days} days)")

                if table == "users":
                    # Erasure, not deletion: scrub the copies first, verify, and
                    # only then remove the account row.
                    outcome = purge_user_erasure(days)
                    count = outcome["deleted"]
                    if outcome["residual"]:
                        raise RuntimeError(
                            f"aborting: {outcome['residual']} rows still hold "
                            f"erased subjects' PII"
                        )
                elif strategy == "hard_delete":
                    count = purge_hard_delete(table, days)
                elif strategy == "anonymize" and table in self.anonymize_map:
                    count = purge_anonymize(table, days, self.anonymize_map[table])
                else:
                    count = 0

                results.append({"table": table, "strategy": strategy, "purged": count})
                icon = "🗑️" if strategy == "hard_delete" else "🔒"
                print(f"  {icon} {count} records processed")

            # Step 4: Record run metadata in Redis
            run_summary = {
                "run_id": self.run_id,
                "completed_at": datetime.now().isoformat(),
                "policies_executed": len(results),
                "total_records_processed": sum(r["purged"] for r in results),
                "results": json.dumps(results)
            }

            self.redis.hset(f"retention:run:{self.run_id}", mapping=run_summary)
            self.redis.set("retention:last_run", self.run_id)
            # Keep run history for 90 days
            self.redis.expire(f"retention:run:{self.run_id}", 90 * 86400)

            # Step 5: Print summary
            print("\n" + "=" * 60)
            print("✅ Retention run complete")
            print(f"   Run ID: {self.run_id}")
            print(f"   Policies: {len(results)}")
            print(f"   Records: {run_summary['total_records_processed']}")

        finally:
            # Always release the lock — and say so honestly if we no longer
            # owned it, because that means our run overran its own timeout.
            if self.release_lock():
                print("🔓 Lock released")
            else:
                print("⚠️ Lock was not ours to release (it expired mid-run)")

        return results

# Run the engine. Note what it *should* find: the manual purge in Step 4 already
# did the work, so a correct daily job on day two removes nothing. Zeros here are
# the pass condition, not a failure.
engine = RetentionEngine()
engine_results = engine.run()

second_pass_total = sum(r["purged"] for r in engine_results)
assert second_pass_total == 0, (
    f"the scheduled run found {second_pass_total} more records to purge right "
    "after Step 4 finished. Either the purge functions are not idempotent (the "
    "anonymize guard is the usual culprit) or the scan and purge disagree about "
    "which rows are expired."
)
print(f"\n✅ Second pass processed {second_pass_total} records — the engine is idempotent")

# The lock must actually exclude a concurrent run.
other = RetentionEngine()
assert other.acquire_lock() is True, "the lock was not released by the finished run"
assert RetentionEngine().acquire_lock() is None, (
    "a second engine acquired the lock while another run holds it"
)
assert other.release_lock() is True, "a run must be able to release its own lock"
print("✅ Lock assertions passed (mutual exclusion + compare-and-delete release)")

In [ ]:
# Show the run history from Redis
r = get_redis_client()

last_run = r.get("retention:last_run")
if last_run:
    run_data = r.hgetall(f"retention:run:{last_run}")
    print("📊 Last Retention Run (from Redis)")
    print("=" * 50)
    for key, value in run_data.items():
        if key == "results":
            results = json.loads(value)
            print(f"  {key}:")
            for r_item in results:
                print(f"    - {r_item['table']}: {r_item['purged']} ({r_item['strategy']})")
        else:
            print(f"  {key}: {value}")

print(f"\n💡 In production, monitoring systems would alert if:")
print(f"   - A retention run fails or doesn't complete")
print(f"   - No run has executed in the past 48 hours")
print(f"   - An unusually large number of records were purged")

## 🎯 Key Takeaways

1. **Define retention before collecting data** — know how long you'll keep it before you start storing it
2. **Three strategies** — hard delete (gone forever), soft delete (grace period), anonymize (keep for analytics)
3. **Audit everything** — the audit log proves compliance to regulators
4. **Automate purging** — manual deletion doesn't scale and is error-prone
5. **Use locks** — prevent duplicate purge runs from corrupting data, and release them with a compare-and-delete so you cannot free someone else's lock
6. **Monitor the engine** — alert if purging stops or behaves unexpectedly
7. **Each policy runs on its own clock** — "one year after resolution" is `resolved_at`, "30-day grace period" is `deletion_requested_at`. Using `created_at` for everything is a different policy that happens to compile, and it silently erases accounts that were still inside their recovery window.
8. **Watch the rows whose clock never starts** — an unresolved ticket from two years ago is kept forever by a policy that reads "one year after resolution". Nothing in the purge will ever see it.
9. **Erasure is not one DELETE** — the same name and address were copied into `orders`, the IP into `activity_log`, PII into ticket free-text. Scrub the copies first, then delete the row, then *verify* no residue is reachable.
10. **Purges must be idempotent** — a policy that re-anonymizes the same rows every night inflates the audit log until the only number a regulator cares about is meaningless.
11. **A count is not proof** — "deleted 6 rows" cannot answer "prove you erased me". The audit entry needs the subject reference, and you need to be able to say why keeping it is lawful.

### What this notebook does *not* do

Everything here happens inside one PostgreSQL database, which is the easy half of
retention. A real erasure has to reach:

- **Backups and snapshots** — usually handled by declaring a maximum backup age and re-applying the erasure on restore, because you cannot rewrite a backup.
- **Read replicas, the WAL, and CDC streams** — deleted rows live on in replication logs until those age out.
- **Search indexes, caches and queues** — Elasticsearch documents, Redis keys, in-flight Kafka messages.
- **Analytics warehouses and ML training sets** — including models already trained on the data.
- **Third-party processors** — every vendor that received the data needs to be told, and you need a record that they confirmed.
- **Legal hold** — an active litigation hold overrides retention, so the engine must check for holds before it deletes anything. Ours does not.
- **Object storage** — invoices as PDFs, uploaded attachments, profile images.

The pattern that makes this tractable is a data inventory (notebook 1's registry)
that lists every place a data element lands, so "erase this person" becomes a
checklist rather than an act of memory.

### What Microsoft Does

- **Azure Data Lifecycle Management** automatically moves data through retention tiers
- **Microsoft 365 Retention Labels** let admins set retention per document
- **Legal Hold** can override retention policies during litigation
- **Data Subject Requests (DSRs)** must complete user deletion within 30 days
- Every Azure service must implement retention as part of its privacy review
- The compliance dashboard shows retention coverage across all services

### What You've Learned in This Series

| Notebook | Topic | Key Skill |
|----------|-------|-----------|
| 1 | Data Classification | Scan and classify PII in databases |
| 2 | Privacy Impact Assessment | Evaluate and score privacy risk before launch |
| 3 | Anonymization Techniques | Protect data with k-anonymity, DP, tokenization |
| 4 | Data Retention & Purging | Automate deletion with audit trails |

These four capabilities form the foundation of **privacy engineering** at any large tech company. Understanding them will help you build systems that respect user privacy and comply with regulations worldwide.